In [1]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel.csv")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [2]:
# ============================================================================
# PPML TWFE Aggregate Score Only
# ============================================================================

# ---- Run Model ----
model_aggregate <- feglm(Vulnerability_count ~ Aggregate_score + log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) 
                        + version_age_days + log1p(Release_count) | package_name + year_month, data = panel,
                         family = "poisson", cluster = ~package_name)

print(model_aggregate)

# ---- Calculate and Print % Change ----
pct_change <- (exp(coef(model_aggregate)["Aggregate_score"]) - 1) * 100

cat("\n------------------------------------------------------------\n")
cat(sprintf("A 1-unit increase in Aggregate_score is associated with a %.2f%% change in Vulnerability count.\n", 
            pct_change))
cat("------------------------------------------------------------\n")


NOTES: 15,965 observations removed because of NA values (RHS: 15,965).
       5,200/0 fixed-effects (22,673 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: Vulnerability_count
Observations: 65,416
Fixed-effects: package_name: 9,056,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error  z value   Pr(>|z|)    
Aggregate_score              -0.030736   0.028291 -1.08645 2.7728e-01    
log1p(downloads_7_day_total)  0.003578   0.002269  1.57719 1.1475e-01    
log1p(dependents)             0.011787   0.005420  2.17464 2.9657e-02 *  
log1p(loc)                    0.194879   0.028884  6.74693 1.5101e-11 ***
version_age_days             -0.000916   0.000317 -2.88569 3.9056e-03 ** 
log1p(Release_count)         -0.022255   0.007492 -2.97061 2.9721e-03 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -238,643.8   Adj. Pseudo R2: 0.809919
           BIC:  578,037.9     Squared Cor.: 0.884782

------------------------------------------------------------
A 1-unit increase in Aggregate_score is associated with a -3.03%

## Lagged Model

In [3]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel.csv")

panel <- panel %>%
  mutate(year_month_date = ym(substr(year_month, 1, 7))) %>%
  arrange(package_name, year_month_date)

# ---- 2. Build a complete monthly grid per package so gaps are explicit ----
panel_complete <- panel %>%
  group_by(package_name) %>%
  complete(year_month_date = seq(min(year_month_date), max(year_month_date), by = "month")) %>%
  ungroup() %>%
  arrange(package_name, year_month_date)

# ---- 3. Lag Aggregate_score by true calendar distance ---------------------
panel_complete <- panel_complete %>%
  group_by(package_name) %>%
  mutate(
    Aggregate_score_lag1 = dplyr::lag(Aggregate_score, n = 1),
    Aggregate_score_lag3 = dplyr::lag(Aggregate_score, n = 3),
    Aggregate_score_lag6 = dplyr::lag(Aggregate_score, n = 6),
    Aggregate_score_lag12 = dplyr::lag(Aggregate_score, n = 12),
  ) %>%
  ungroup()

# ---- 4. Drop synthetic gap-filler rows — keep only real observations -----
panel_final <- panel_complete %>%
  semi_join(panel, by = c("package_name", "year_month_date"))

# ============================================================================
# PPML TWFE — Lagged Aggregate Score Models (1, 3, 6 months)
# ============================================================================

model_lag1 <- feglm(
  Vulnerability_count ~ Aggregate_score_lag1 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

model_lag3 <- feglm(
  Vulnerability_count ~ Aggregate_score_lag3 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

model_lag6 <- feglm(
  Vulnerability_count ~ Aggregate_score_lag6 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

model_lag12 <- feglm(
  Vulnerability_count ~ Aggregate_score_lag12 +
    log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) +
    version_age_days + log1p(Release_count) |
    package_name + year_month,
  data = panel_final, family = "poisson", cluster = ~package_name
)

# ---- 5. Combined summary table --------------------------------------------
lag_models <- list(
  "1-month lag" = model_lag1,
  "3-month lag" = model_lag3,
  "6-month lag" = model_lag6,
  "12-month lag" = model_lag12
)

modelsummary(lag_models, stars = TRUE)

# ---- 6. % change and effective N for each lag ------------------------------
for (name in names(lag_models)) {
  m <- lag_models[[name]]
  coef_name <- names(coef(m))[1]  # the Aggregate_score_lagX term
  pct_change <- (exp(coef(m)[coef_name]) - 1) * 100
  n_obs <- nobs(m)
  cat("\n------------------------------------------------------------\n")
  cat(sprintf("%s (N = %d): A 1-unit increase in Aggregate Score (%s) is associated with a %.2f%% change in Vulnerability_count.\n",
              name, n_obs, coef_name, pct_change))
  cat("------------------------------------------------------------\n")
}

NOTES: 61,429 observations removed because of NA values (RHS: 61,429).
       4,272/0 fixed-effects (10,623 observations) removed because of only 0 outcomes or singletons.

NOTES: 66,943 observations removed because of NA values (RHS: 66,943).
       3,804/0 fixed-effects (9,363 observations) removed because of only 0 outcomes or singletons.

NOTES: 73,782 observations removed because of NA values (RHS: 73,782).
       3,462/0 fixed-effects (8,029 observations) removed because of only 0 outcomes or singletons.

NOTES: 84,730 observations removed because of NA values (RHS: 84,730).
       3,213/0 fixed-effects (6,031 observations) removed because of only 0 outcomes or singletons.




+------------------------------+-------------+-------------+-------------+--------------+
|                              | 1-month lag | 3-month lag | 6-month lag | 12-month lag |
+==============================+=============+=============+=============+==============+
| Aggregate_score_lag1         | -0.020      |             |             |              |
+------------------------------+-------------+-------------+-------------+--------------+
|                              | (0.027)     |             |             |              |
+------------------------------+-------------+-------------+-------------+--------------+
| log1p(downloads_7_day_total) | 0.000       | 0.002       | -0.003      | -0.007       |
+------------------------------+-------------+-------------+-------------+--------------+
|                              | (0.003)     | (0.003)     | (0.004)     | (0.006)      |
+------------------------------+-------------+-------------+-------------+--------------+
| log1p(d


------------------------------------------------------------
1-month lag (N = 32002): A 1-unit increase in Aggregate Score (Aggregate_score_lag1) is associated with a -1.96% change in Vulnerability_count.
------------------------------------------------------------

------------------------------------------------------------
3-month lag (N = 27748): A 1-unit increase in Aggregate Score (Aggregate_score_lag3) is associated with a 2.31% change in Vulnerability_count.
------------------------------------------------------------

------------------------------------------------------------
6-month lag (N = 22243): A 1-unit increase in Aggregate Score (Aggregate_score_lag6) is associated with a 0.28% change in Vulnerability_count.
------------------------------------------------------------

------------------------------------------------------------
12-month lag (N = 13293): A 1-unit increase in Aggregate Score (Aggregate_score_lag12) is associated with a 3.31% change in Vulnerability_c